# LLaDA Sudoku on USC CARC


Run this notebook from the `sean-carc` branch in a CARC OnDemand JupyterLab GPU session. Select the Discovery `gpu` partition and request one BF16-capable GPU with more than 20 GiB of memory (A100, A40, or L40S), then select the dedicated `DLM CARC` kernel.

The first session downloads approximately 16 GB of pinned model files into `/project2/xiangren_1987/group_37/model-cache`. Later sessions reuse that snapshot. The dataset cache lives in `/project2/xiangren_1987/group_37/datasets`, while experiment artifacts are saved under your home directory. Only run configuration receipts use the compute node's temporary directory.

Before starting JupyterLab, clone the repository on CARC and register the existing `/project2/xiangren_1987/group_37/envs/dlm` environment as the Jupyter kernel described below. No GitHub credentials are read from this notebook.

The Config Updater starts with one puzzle and inspectable MCTS initialization. Select `baseline` to use the original sampler. The separate iLLaDA notebook retains its baseline defaults. Evaluation is model-only: Sudoku rules validate answers, without repairing them.


In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Launch CARC OnDemand JupyterLab on the Discovery gpu partition with one GPU.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Request an A100, A40, or L40S with more than 20 GiB of GPU memory.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


GPU: NVIDIA L40S | GPU memory: 44.4 GiB


## 1. Locate the CARC checkout
Clone and switch to `sean-carc` from a CARC shell before opening this notebook. The cell below locates the repository from the Jupyter working directory and records the exact commit; it never fetches, switches branches, or modifies the checkout.


In [3]:
from pathlib import Path
import os
import subprocess
import sys

EXPECTED_BRANCH = "sean-carc"
REPO_DIR_OVERRIDE = None  # Example: Path.home() / "CSCI544-DLM"

def find_repository(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "src/dlm_project").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the CSCI544-DLM checkout on CARC.")

REPO_DIR = Path(REPO_DIR_OVERRIDE).expanduser().resolve() if REPO_DIR_OVERRIDE else find_repository(Path.cwd())
if not (REPO_DIR / ".git").exists() or not (REPO_DIR / "src/dlm_project").is_dir():
    raise RuntimeError(f"Invalid repository path: {REPO_DIR}")

def local_git(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

branch = local_git("branch", "--show-current")
if branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Expected branch {EXPECTED_BRANCH!r}, found {branch!r}. Switch branches from a shell, then restart the kernel.")
print(f"Code ready: {REPO_DIR} | branch: {branch} | commit: {local_git('rev-parse', '--short', 'HEAD')}")


Code ready: /home1/xchen664/CSCI544-DLM | branch: sean-carc | commit: 44b97da


## 2. Verify the CARC Python environment
Use the existing `/project2/xiangren_1987/group_37/envs/dlm` environment. From an allocated CARC GPU compute session, run the following once from the repository root to install the pinned project dependencies and register the kernel:

```bash
module purge
module load conda
eval "$(conda shell.bash hook)"
export DLM_ENV=/project2/xiangren_1987/group_37/envs/dlm
conda activate "$DLM_ENV"
python -c 'import sys, torch; print(sys.version); print(torch.__version__, torch.cuda.is_available())'
python -m pip install ipykernel pandas
python -m pip install -r requirements/llada-colab.txt -r requirements/dev.txt
python -m pip install --no-deps -e .
python -m ipykernel install --user --name dlm-carc --display-name "DLM CARC"
```

Select that kernel when opening this notebook. LLaDA requires Transformers 4.38.2; the check below fails early if the wrong environment is selected.


In [4]:
try:
    import dlm_project
    import datasets
    import transformers
except ModuleNotFoundError as error:
    raise RuntimeError("Select the prepared DLM CARC kernel; do not install dependencies inside the notebook session.") from error

if transformers.__version__ != "4.38.2":
    raise RuntimeError(f"Expected Transformers 4.38.2, found {transformers.__version__}.")
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)


/project2/xiangren_1987/group_37/envs/dlm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]

Python: /project2/xiangren_1987/group_37/envs/dlm/bin/python
PyTorch: 2.14.1+cu130 | Transformers: 4.38.2 | Datasets: 5.0.1


### Check the search on CUDA before loading LLaDA
The model forward pass, softmax, entropy, top-token filtering and confidence factors run on the **GPU**. Tree states, UCB bookkeeping, final sorting of compact action statistics, and notebook tables stay on the **CPU**. This keeps branch copies out of GPU memory.

Run these three small CUDA tests first. They compare CPU and CUDA results for positive, negative and zero-entropy rewards; verify cached completion; and check that every model input reaches CUDA. They use tiny controlled distributions, so they do not download weights. The subprocess explicitly requires CUDA and fails if it is unavailable.

The later inspection and generation cells use the real HF LLaDA model with `device="cuda"` and BF16 weights. Each subprocess loads one model, and the kernel only displays saved traces.


In [5]:
cuda_checks = """
import torch
import pytest

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in the test subprocess. Select a GPU runtime.")
print("CUDA test device:", torch.cuda.get_device_name(0), flush=True)
raise SystemExit(pytest.main(["-q", "tests/decoding/test_mcts_cuda.py"]))
"""
subprocess.run([sys.executable, "-u", "-c", cuda_checks], cwd=REPO_DIR, check=True)


CUDA test device: NVIDIA L40S
...                                                                      [100%]
3 passed in 0.46s


CompletedProcess(args=['/project2/xiangren_1987/group_37/envs/dlm/bin/python', '-u', '-c', '\nimport torch\nimport pytest\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable in the test subprocess. Select a GPU runtime.")\nprint("CUDA test device:", torch.cuda.get_device_name(0), flush=True)\nraise SystemExit(pytest.main(["-q", "tests/decoding/test_mcts_cuda.py"]))\n'], returncode=0)

## 3. Configure persistent CARC storage
The pinned model is stored directly in `/project2/xiangren_1987/group_37/model-cache`, the Hugging Face dataset cache in `/project2/xiangren_1987/group_37/datasets`, and experiment outputs under `~/LLaDA/Sudoku`. Allow at least **20 GB** in the model directory plus room for the dataset cache.

`STAGE_MODEL_TO_NODE = False` loads directly from `/project2` and avoids a second 16 GB copy. Set it to `True` only when the allocated node has sufficient temporary disk and repeated local model loads justify the copy. Existing project files are never moved or deleted.


In [6]:
import importlib

PERSISTENT_MODEL_DIR = Path("/project2/xiangren_1987/group_37/model-cache")
DATASET_CACHE_DIR = Path("/project2/xiangren_1987/group_37/datasets")
HUGGINGFACE_HOME = PERSISTENT_MODEL_DIR / ".huggingface"
HUB_CACHE_DIR = HUGGINGFACE_HOME / "hub"
EXPERIMENT_ROOT = Path.home()
SESSION_BASE = Path(os.environ.get("TMPDIR", "/tmp"))
SESSION_ID = os.environ.get("SLURM_JOB_ID", str(os.getpid()))
SESSION_ROOT = SESSION_BASE / f"csci544-dlm-{SESSION_ID}"
MODEL_FAMILY = "llada"

from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
RUNTIME_MODEL_DIR = SESSION_ROOT / "models" / MODEL_NAME
STAGE_MODEL_TO_NODE = False
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

for directory in (PERSISTENT_MODEL_DIR, DATASET_CACHE_DIR, HUB_CACHE_DIR, EXPERIMENT_ROOT, SESSION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
for directory in (PERSISTENT_MODEL_DIR, DATASET_CACHE_DIR, EXPERIMENT_ROOT):
    if not os.access(directory, os.W_OK):
        raise PermissionError(f"Directory is not writable: {directory}")

os.environ["HF_HOME"] = str(HUGGINGFACE_HOME)
os.environ["HF_HUB_CACHE"] = str(HUB_CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
MODEL_PATH = RUNTIME_MODEL_DIR if STAGE_MODEL_TO_NODE else PERSISTENT_MODEL_DIR
print("Persistent model:", PERSISTENT_MODEL_DIR)
print("Dataset cache:", DATASET_CACHE_DIR)
print("Experiment root:", EXPERIMENT_ROOT / profile["label"] / "Sudoku")
print("Session temporary directory:", SESSION_ROOT)


Persistent model: /project2/xiangren_1987/group_37/model-cache
Dataset cache: /project2/xiangren_1987/group_37/datasets
Experiment root: /home1/xchen664/LLaDA/Sudoku
Session temporary directory: /tmp/SLURM_13172833/csci544-dlm-13172833


## 4. Load this run's base configuration
Set the model, cache locations, and timezone. Choose sample count and generation parameters in the **Config Updater** below, which writes the final configuration and checks the data before inference.


In [7]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/sudoku/llada_mcts.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["experiment_root"] = str(EXPERIMENT_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE
RUN_CONFIG = SESSION_ROOT / "llada-sudoku-run.yaml"
RUNNER = REPO_DIR / "scripts/run_sudoku.py"
print("Model revision:", profile["revision"])


Model revision: 08b83a6feb34df1a6011b80c3c00c7563e963b07


## 5. Cache the model under `/project2`
Download the pinned model's tokenizer, chat template, configuration, Python implementation, and all shards listed in its weight index. Later CARC sessions reuse the same files; keep the model folder's hidden `.cache` directory. The cache script's `--drive-dir` name is historical—the argument accepts the persistent `/project2` directory.

Rerun this cell after an interrupted download. If validation reports missing files or insufficient disk space, resolve that before inference. No weights are committed to GitHub.


In [8]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(PERSISTENT_MODEL_DIR)]
if STAGE_MODEL_TO_NODE:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


Model ready: /project2/xiangren_1987/group_37/model-cache


Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 231.89it/s]


CompletedProcess(args=['/project2/xiangren_1987/group_37/envs/dlm/bin/python', '-u', '/home1/xchen664/CSCI544-DLM/scripts/cache_model.py', '--model', 'llada', '--drive-dir', '/project2/xiangren_1987/group_37/model-cache'], returncode=0)

## Config Updater
Start with one puzzle, 256 output positions, temperature 1.0, Lc=20, C=3, K1=3 and K2=5. Each simulation evaluates only its immediate child and uses the parent-to-child entropy gain. Search costs one root forward plus one forward per child; candidate ranking uses cached rewards. Explicit budget mode limits search. Sudoku uses disjoint synthetic demonstrations for inline three-step reasoning, preserving its final answer format.


In [9]:
NUM_SAMPLES = 1
DECODING_METHOD = "mcts_entropy"  # baseline | confidence | mcts_entropy
config["data"]["num_samples"] = NUM_SAMPLES
config["generation"].update(
    gen_length=256,
    steps=256,
    block_length=256,
    temperature=1.0 if DECODING_METHOD == "mcts_entropy" else 0.0,
    remasking="low_confidence",
)
config["decoding"] = {
    "method": DECODING_METHOD,
    "seed": 1,
    "mcts": {
        "init_tokens": 20,       # Lc
        "num_candidates": 3,    # C
        "top_k_tokens": 3,      # K1
        "top_k_actions": 5,     # K2
        "max_model_calls": None,  # immediate-child evaluations; no default cap
        "margin_gamma": 10.0,
        "exploration_c": 0.01,
        "epsilon": 1e-8,
    },
}

exp_name = experiment_paths.experiment_name(config["generation"], config["decoding"])
config["output_dir"] = str(experiment_paths.experiment_parent(config))
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(f"Experiment output directory: {config['output_dir']}\n")
print(RUN_CONFIG.read_text())
subprocess.run([sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG), "--dataset-only"], check=True)


Experiment output directory: /home1/xchen664/LLaDA/Sudoku/gen256-steps256-blk256-temp1-lowconf-mcts_entropy-gamma10-eps1e-08-seed1-lc20-c3-k1_3-k2_5-budgetnone-ucb0.01-medal-v2

data:
  num_samples: 1
  cache_dir: /project2/xiangren_1987/group_37/datasets
model:
  family: llada
  model_path: /project2/xiangren_1987/group_37/model-cache
  device: cuda
  dtype: bfloat16
  local_files_only: true
generation:
  steps: 256
  gen_length: 256
  block_length: 256
  temperature: 1.0
  remasking: low_confidence
decoding:
  method: mcts_entropy
  seed: 1
  mcts:
    init_tokens: 20
    num_candidates: 3
    top_k_tokens: 3
    top_k_actions: 5
    max_model_calls: null
    margin_gamma: 10.0
    exploration_c: 0.01
    epsilon: 1.0e-08
experiment_root: /home1/xchen664
experiment_timezone: America/Los_Angeles
output_dir: /home1/xchen664/LLaDA/Sudoku/gen256-steps256-blk256-temp1-lowconf-mcts_entropy-gamma10-eps1e-08-seed1-lc20-c3-k1_3-k2_5-budgetnone-ucb0.01-medal-v2

Loaded 1 puzzle(s). First promp

CompletedProcess(args=['/project2/xiangren_1987/group_37/envs/dlm/bin/python', '-u', '/home1/xchen664/CSCI544-DLM/scripts/run_sudoku.py', '--config', '/tmp/SLURM_13172833/csci544-dlm-13172833/llada-sudoku-run.yaml', '--dataset-only'], returncode=0)

## Inspect one expansion
Run this section with `DECODING_METHOD = "mcts_entropy"`. One search cycle expands up to five actions and performs one forward per immediate child. Tree states keep only their initialized tokens. The trace records parent-to-child entropy gains and one backup per evaluated child.


In [10]:
import json
import uuid
import pandas as pd
from IPython.display import display


def read_trace(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


def show_mcts_cycle(events, cycle):
    current = [event for event in events if event.get("cycle") == cycle]
    if not current:
        raise ValueError(f"No cycle {cycle}; available: {sorted({e['cycle'] for e in events if 'cycle' in e})}")
    print(f"Cycle {cycle} | cumulative search forwards: {current[-1]['search_model_calls']}")
    selection = [event for event in current if event["stage"] == "selection"]
    print("1. Selection")
    for event in selection:
        print("Traversed node IDs:", event["path"])
        rows = [{"parent_id": decision["parent_id"], "chosen_id": decision["chosen_id"], **sibling}
                for decision in event["decisions"] for sibling in decision["siblings"]]
        if rows:
            display(pd.DataFrame(rows))
        else:
            print("Root is unexpanded: no sibling UCB comparison yet.")
    print("2. Expansion — each row is an independent child")
    display(pd.DataFrame([{"parent_id": e["parent_id"], "child_id": e["child_id"],
                           "depth": e["depth"], **e["action"]}
                          for e in current if e["stage"] == "expansion"]))
    print("3. One-step evaluation")
    evaluations = [e for e in current if e["stage"] == "evaluation"]
    columns = ["parent_id", "child_id", "entropy_before", "entropy_after",
               "committed_position_entropy", "reward", "context_gain", "search_model_calls"]
    display(pd.DataFrame([{key: e[key] for key in columns} for e in evaluations]))
    for e in evaluations:
        print(f"Child {e['child_id']}: {e['child_state']['partial_output']}")
    print("4. Backpropagation — child to root for each evaluated action")
    rows = []
    for event in current:
        if event["stage"] == "backpropagation":
            for update in event["updates"]:
                rows.append({"evaluated_child": event["child_id"], "ancestor_id": update["node_id"],
                             **{f"{key}_before": value for key, value in update["before"].items()},
                             **{f"{key}_after": value for key, value in update["after"].items()}})
    display(pd.DataFrame(rows))
    print("Per-position entropy (0 at committed positions)")
    if evaluations:
        display(pd.DataFrame({"position": range(len(evaluations[0]["entropies_before"])),
                              "parent": evaluations[0]["entropies_before"],
                              **{f"child_{e['child_id']}": e["entropies_after"] for e in evaluations}}))
    candidates = [e for e in current if e["stage"] == "candidate"]
    if candidates:
        print("Collected candidates (duplicate states do not increment the count)")
        display(pd.DataFrame(candidates))


In [11]:
if config["decoding"]["method"] != "mcts_entropy":
    raise ValueError("Select mcts_entropy and rerun Config Updater for inspection.")
INSPECTION_DIR = EXPERIMENT_ROOT / profile["label"] / "Sudoku" / "inspections" / uuid.uuid4().hex
INSPECTION_DIR.mkdir(parents=True, exist_ok=False)
INSPECTOR = REPO_DIR / "scripts/inspect_mcts.py"
ONE_STEP_TRACE = INSPECTION_DIR / "one_step.jsonl"
subprocess.run([
    sys.executable, "-u", str(INSPECTOR), "--phase", "one_step",
    "--config", str(RUN_CONFIG), "--trace-path", str(ONE_STEP_TRACE),
], check=True)
one_step_events = read_trace(ONE_STEP_TRACE)
assert torch.device(one_step_events[-1]["model_device"]).type == "cuda"
print("HF model device:", one_step_events[-1]["model_device"])
print("Search-state storage:", one_step_events[-1]["search_state_device"])
root_event = next(e for e in one_step_events if e["stage"] == "initialization")
print("Root retained actions — actual HF model predictions")
display(pd.DataFrame(root_event["actions"]))
show_mcts_cycle(one_step_events, 1)
print("Inspection summary:")
display(pd.DataFrame([one_step_events[-1]]).drop(columns=["selected_state"]))


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 6/6 [00:18<00:00,  3.09s/it]


{
  "implementation_version": "medal_reconstruction_v2",
  "paper_url": "https://aclanthology.org/2026.findings-eacl.180.pdf",
  "reconstruction_assumptions": {
    "simulation": "one forward on the immediate child; cache its entropy and actions",
    "reward": "Eq9 gain from cached parent entropy to immediate child entropy",
    "candidate_evaluation": "rank cached mean tree rewards; no additional model forwards",
    "completion": "fresh partial-state completion; literal softmax(s/temperature) over all legal position-token actions",
    "decomposition": "inline three subtasks; disjoint two-shot demonstrations",
    "quota": "stop immediately at first C distinct depth-Lc states",
    "constants": "gamma=10 and exploration_c=0.01 are repository defaults, not verified author settings"
  },
  "evaluation_variant": "MEDAL reconstruction (immediate-child IG)",
  "finished": false,
  "stop_reason": null,
  "stop_mode": "candidates",
  "search_model_calls": 6,
  "tree_analysis_model_calls": 

,output_position,token_id,confidence_score,probability,entropy,margin,decoded_token
0,255,126081,0.994850,0.999500,0.004617,0.999041,<|endoftext|>
1,254,126348,0.652650,0.903924,0.325397,0.808651,<|eot_id|>
2,1,13,0.070080,0.578196,2.100698,0.464343,.
3,254,126081,0.068789,0.095273,0.325397,0.808651,<|endoftext|>
4,0,16,0.061910,0.579773,2.230321,0.501309,1


Cycle 1 | cumulative search forwards: 6
1. Selection
Traversed node IDs: [0]
Root is unexpanded: no sibling UCB comparison yet.
2. Expansion — each row is an independent child


,parent_id,child_id,depth,output_position,token_id,confidence_score,probability,entropy,margin,decoded_token
0,0,1,1,255,126081,0.994850,0.999500,0.004617,0.999041,<|endoftext|>
1,0,2,1,254,126348,0.652650,0.903924,0.325397,0.808651,<|eot_id|>
2,0,3,1,1,13,0.070080,0.578196,2.100698,0.464343,.
3,0,4,1,254,126081,0.068789,0.095273,0.325397,0.808651,<|endoftext|>
4,0,5,1,0,16,0.061910,0.579773,2.230321,0.501309,1


3. One-step evaluation


... (3 outputs left out)

Per-position entropy (0 at committed positions)


,position,parent,child_1,child_2,child_3,child_4,child_5
0,0,2.230321,1.884409,2.029075e+00,0.107861,1.862927,0.000000
1,1,2.100698,1.831056,1.964820e+00,0.000000,1.441714,0.405895
2,2,3.930009,4.023752,4.006239e+00,2.551835,3.996442,2.970732
3,3,3.441371,3.223845,3.331506e+00,1.425732,3.127136,1.811901
4,4,3.584983,3.278325,3.592801e+00,2.638584,3.015680,2.877763
...,...,...,...,...,...,...,...
251,251,2.571689,2.520840,2.459736e+00,2.494738,0.090999,2.442046
252,252,2.513769,2.486501,2.506630e+00,2.394760,0.118258,2.302810
253,253,2.702467,2.555553,2.530879e+00,2.510773,0.051138,2.440472
254,254,0.325397,0.539909,0.000000e+00,0.462341,0.000000,0.579436


Inspection summary:


,stage,phase,implementation_version,paper_url,reconstruction_assumptions,evaluation_variant,finished,stop_reason,stop_mode,search_model_calls,...,selected_score,fallback_used,fallback_reason,budget_exhausted,candidate_evaluations,model_device,search_state_device,completion_model_calls,latency_seconds,peak_gpu_memory_bytes
0,inspection,one_step,medal_reconstruction_v2,https://aclanthology.org/2026.findings-eacl.18...,{'simulation': 'one forward on the immediate c...,MEDAL reconstruction (immediate-child IG),False,None,candidates,6,...,0.268207,False,None,False,[],cuda:0,cpu,0,1.307483,16563983872


## Inspect initialization
This fresh subprocess continues all MCTS cycles until it collects at least `C` distinct depth-`Lc` states, exhausts the frontier, or reaches the search-forward cap. Target-depth nodes are closed. Candidate mode stops as soon as `C` distinct candidates are collected.

Change only `CYCLE_TO_VIEW` to inspect another cycle without loading the model again. Candidates are ranked by cached mean tree rewards; at these leaves, that equals the immediate parent-to-child entropy gain. Strict candidate mode fails if fewer than `C` candidates are reached. Explicit budget mode ranks any collected candidates, or falls back to the deepest evaluated state if none exists.


In [ ]:
SEARCH_TRACE = INSPECTION_DIR / "search.jsonl"
subprocess.run([
    sys.executable, "-u", str(INSPECTOR), "--phase", "search",
    "--config", str(RUN_CONFIG), "--trace-path", str(SEARCH_TRACE),
], check=True)
search_events = read_trace(SEARCH_TRACE)
print("Available cycles:", sorted({e["cycle"] for e in search_events if "cycle" in e}))
summary = search_events[-1]
assert torch.device(summary["model_device"]).type == "cuda"
print("Selected partial state:", summary["selected_state"]["partial_output"])
display(pd.DataFrame([{k: v for k, v in summary.items() if k != "selected_state"}]))


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 6/6 [00:24<00:00,  4.10s/it]


In [ ]:
CYCLE_TO_VIEW = 1
show_mcts_cycle(search_events, CYCLE_TO_VIEW)


## 6. Run full MEDAL generation
The subprocess loads the model once, performs immediate-child MCTS and cached candidate ranking, then completes the selected partial state using literal adjusted-score softmax. CPU trees and GPU model calls remain separately inspectable. Final completion uses a fresh distribution rather than a cached first commitment.


In [ ]:
import json
import uuid

latest = None
# A unique receipt identifies this invocation; it cannot select an older run.
RUN_MANIFEST = SESSION_ROOT / f"{MODEL_FAMILY}-run-{uuid.uuid4().hex}.json"
subprocess.run([
    sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG),
    "--run-manifest", str(RUN_MANIFEST),
], check=True)
receipt = json.loads(RUN_MANIFEST.read_text())
latest = Path(receipt["run_dir"])
print("This run:", latest)


## 7. Inspect all saved predictions and plots
All artifacts are in the current experiment's timestamp folder: `config.json`, `metrics.json`, `predictions.jsonl`, `traces/puzzle_000.jsonl`, and `plots/puzzle_000.png`, etc.

The displayed images are the PNGs saved during evaluation. Incomplete answers keep their original row positions, complete invalid grids are flagged, and red digits conflict with given clues. Nothing is repaired by a solver.


In [ ]:
if latest is None:
    raise RuntimeError("Complete the run cell before inspecting its results.")
print("Results:", latest)
print(json.dumps(json.loads((latest / "metrics.json").read_text()), indent=2))
print("\n--- All Predictions ---")
with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        record = json.loads(line)
        print(f"\nPrediction for Puzzle Index {record['index']}:")
        print(json.dumps(record, indent=2))


### Verify the actual generation trace on CARC
This checks the saved HF inference artifacts, forward accounting, and preservation of search commitments. It displays the raw prediction, existing Sudoku scores, and measured GPU cost. A failed Sudoku answer remains a valid decoder execution; quality is reported separately.


In [ ]:
records = [json.loads(line) for line in (latest / "predictions.jsonl").read_text().splitlines()]
assert len(records) == config["data"]["num_samples"]
PUZZLE_INDEX = 0
record = records[PUZZLE_INDEX]
stats = record["decoding_stats"]
print("Actual HF output:", record["raw_output"])
display(pd.DataFrame([{key: record[key] for key in ["parse_success", "clues_preserved", "valid_sudoku", "exact", "correct_cells"]}]))
display(pd.DataFrame([stats]))
assert torch.device(stats["model_device"]).type == "cuda", "The loaded HF model must be on CUDA."
assert stats["peak_gpu_memory_bytes"] > 0, "Run inference in a CARC OnDemand GPU session."
assert stats["total_model_calls"] == stats["search_model_calls"] + stats["completion_model_calls"]
if stats["method"] == "mcts_entropy":
    generation_events = read_trace(latest / record["trace_file"])
    evaluations = [e for e in generation_events if e["stage"] == "evaluation"]
    assert stats["search_model_calls"] == sum(stats[f"{phase}_model_calls"]
        for phase in ("tree_analysis", "simulation", "candidate_evaluation"))
    cap = config["decoding"]["mcts"]["max_model_calls"]
    assert cap is None or stats["search_model_calls"] <= cap
    selected = next(e for e in generation_events if e["stage"] == "termination")["selected_state"]
    completions = [e for e in generation_events if e["stage"] == "rollout_step" and e.get("phase") == "completion"]
    positions = [c["output_position"] for c in selected["committed"]]
    positions += [e["action"]["output_position"] for e in completions]
    assert len(set(positions)) == len(positions) == config["generation"]["gen_length"]
    assert stats["completion_model_calls"] == len(completions)
    assert not completions or completions[-1]["remaining_masks"] == 0
    show_mcts_cycle(generation_events, globals().get("CYCLE_TO_VIEW", 1))
print("GPU artifact checks passed; see Sudoku quality metrics above and saved plots below.")


In [ ]:
from IPython.display import Image, display

with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        current_record = json.loads(line)
        plot_path = latest / current_record["plot_file"]
        print(f"Puzzle Index {current_record['index']} | {plot_path}")
        display(Image(filename=str(plot_path)))


## 8. Verify caching, then increase the sample count
After the one-puzzle verification, set `NUM_SAMPLES = 3` in the Config Updater and rerun it and the full-generation/results cells. To check reuse, rerun step 5 and evaluation: the pinned `/project2` snapshot is reused, and a new experiment timestamp is created.

Review Sudoku validity and clue preservation before increasing to 10 or 100 puzzles. Change parameters only in the Config Updater; each run saves the resolved configuration. Existing result folders under `~/LLaDA/Sudoku` remain untouched. For longer experiments, prefer `scripts/run_sudoku.py` in a Slurm batch job so the run does not depend on an interactive Jupyter session.

### Notes and references
- Fixed-length masked generation uses the LLaDA checkpoint's own tokenizer and chat template. There is no training, solver repair, or automatic variable-length decoding.
- Exact accuracy compares all 81 digits with the reference. Cell accuracy divides matching cells by `81 × processed puzzles`. Failed parsing produces an empty prediction and zero matching cells.
- Completion, clue-preservation, and Sudoku-validity rates use processed puzzles as their denominator. A complete grid may still be invalid.
- Config metadata records the model identity/revision, Git commit, run time, and timezone. Saved notebook outputs are cleared in Git; run the cells to obtain new results.
- [Model](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct) · [Upstream sampler](https://github.com/ML-GSAI/LLaDA) · [Dataset](https://huggingface.co/datasets/certen/Sudoku-1M-Puzzles)
